# IMDN x3 and x4 Colab Pilot

This notebook extends the successful IMDN x2 pilot to x3 and x4 using the same `baby.png` image. It verifies both official checkpoints, measures T4 GPU inference, records native-versus-target dimensions, calculates the fixed project metrics, and saves all outputs to Google Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a GPU runtime and reconnect.')
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
from app.deep_learning.checkpoints import (
    IMDN_CHECKPOINTS,
    download_official_imdn_checkpoint,
)

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
checkpoint_paths = {
    scale: download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale)
    for scale in (3, 4)
}
for scale, checkpoint_path in checkpoint_paths.items():
    print(f'IMDN x{scale} verified: {checkpoint_path}')
    print('SHA-256:', IMDN_CHECKPOINTS[scale].sha256)

In [ ]:
from app.config import dataset_hr_directory, dataset_lr_directory
from app.evaluation.images import load_rgb_image, pair_image_paths

def load_baby_pair(scale):
    pairs = pair_image_paths(
        dataset_hr_directory('Set5', DATA_ROOT),
        dataset_lr_directory('Set5', scale, DATA_ROOT),
    )
    try:
        hr_path, lr_path = next(
            pair for pair in pairs if pair[0].name.lower() == 'baby.png'
        )
    except StopIteration as error:
        raise RuntimeError('baby.png is missing from the prepared Set5 pairs.') from error
    return hr_path, lr_path, load_rgb_image(hr_path), load_rgb_image(lr_path)

for scale in (3, 4):
    _, _, hr_image, lr_image = load_baby_pair(scale)
    print(
        f'x{scale}: LR={lr_image.size}, native={tuple(v * scale for v in lr_image.size)}, '
        f'HR={hr_image.size}'
    )

In [ ]:
import json
from dataclasses import asdict
from datetime import UTC, datetime
from statistics import mean, median

from app.deep_learning.alignment import align_reconstruction_to_target
from app.deep_learning.imdn import (
    load_pretrained_imdn,
    pil_to_tensor,
    tensor_to_pil,
)
from app.evaluation.metrics import calculate_quality_metrics

def run_pilot(scale):
    hr_path, _, reference_hr, lr_image = load_baby_pair(scale)
    model = load_pretrained_imdn(checkpoint_paths[scale], scale, DEVICE)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    input_tensor = pil_to_tensor(lr_image, DEVICE)

    with torch.inference_mode():
        for _ in range(3):
            model(input_tensor)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    latencies_ms = []
    with torch.inference_mode():
        for _ in range(10):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            output_tensor = model(input_tensor)
            end.record()
            torch.cuda.synchronize()
            latencies_ms.append(start.elapsed_time(end))

    native_reconstruction = tensor_to_pil(output_tensor)
    expected_native_size = (lr_image.width * scale, lr_image.height * scale)
    if native_reconstruction.size != expected_native_size:
        raise RuntimeError(
            f'IMDN x{scale} returned {native_reconstruction.size}; '
            f'expected {expected_native_size}.'
        )
    aligned = align_reconstruction_to_target(native_reconstruction, reference_hr.size)
    metrics = calculate_quality_metrics(reference_hr, aligned.image, border=scale)
    peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2

    output_directory = DATA_ROOT / 'results' / 'phase3' / 'pilot' / f'imdn_x{scale}'
    output_directory.mkdir(parents=True, exist_ok=True)
    image_output_path = output_directory / f'{hr_path.stem}_imdn_x{scale}.png'
    aligned.image.save(image_output_path)
    record = {
        'dataset': 'Set5',
        'image': hr_path.name,
        'scale': f'x{scale}',
        'method': 'imdn',
        'device': torch.cuda.get_device_name(0),
        'parameter_count': parameter_count,
        'latency_mean_ms': mean(latencies_ms),
        'latency_median_ms': median(latencies_ms),
        'peak_gpu_memory_mb': peak_memory_mb,
        'native_width': aligned.native_size[0],
        'native_height': aligned.native_size[1],
        'target_width': aligned.target_size[0],
        'target_height': aligned.target_size[1],
        'dimension_adjusted': aligned.dimension_adjusted,
        'dimension_policy': 'native_model_grid_then_full_target_size_adjustment',
        **metrics,
        'checkpoint': asdict(IMDN_CHECKPOINTS[scale]),
        'generated_at_utc': datetime.now(UTC).isoformat(),
    }
    record_output_path = output_directory / f'{hr_path.stem}_imdn_x{scale}_pilot.json'
    record_output_path.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
    return record, lr_image, aligned.image, reference_hr, image_output_path, record_output_path

pilot_results = {scale: run_pilot(scale) for scale in (3, 4)}

In [ ]:
for scale in (3, 4):
    record, _, _, _, image_path, record_path = pilot_results[scale]
    print(f'\nIMDN x{scale}')
    print('Parameters:', f"{record['parameter_count']:,}")
    print('Mean GPU latency (ms):', round(record['latency_mean_ms'], 3))
    print('Median GPU latency (ms):', round(record['latency_median_ms'], 3))
    print('Peak GPU memory (MB):', round(record['peak_gpu_memory_mb'], 2))
    print(
        'Dimensions:',
        (record['native_width'], record['native_height']),
        '->',
        (record['target_width'], record['target_height']),
        'adjusted=',
        record['dimension_adjusted'],
    )
    print(json.dumps({key: record[key] for key in ('psnr_y', 'ssim_y', 'psnr_rgb', 'ssim_rgb')}, indent=2))
    print('Saved image:', image_path)
    print('Saved pilot record:', record_path)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for row, scale in enumerate((3, 4)):
    _, lr_image, reconstruction, reference_hr, _, _ = pilot_results[scale]
    for axis, image, title in zip(
        axes[row],
        (lr_image, reconstruction, reference_hr),
        (f'Prepared LR x{scale}', f'IMDN x{scale}', 'Reference HR'),
    ):
        axis.imshow(image)
        axis.set_title(title)
        axis.axis('off')
plt.tight_layout()
plt.show()

## Completion test

Both scales pass when their checksums verify, parameter counts print, native dimensions are reported, four metrics are calculated, and two images plus two JSON records are saved. The expected `baby.png` behavior is an x3 target-size adjustment and no x4 adjustment.